# UCSD Greynet Telescope — pcap → FlowTuple, and a month of empirical analysis

`greynet_data_access.ipynb` found that `telescope-sdscnt-pcap-live` holds **raw packet
captures** (`gzip -> pcap (usec LE)`), not FlowTuple, e.g.

```
datasource=sdsc-nt/year=2025/month=02/day=20/hour=20/sdscnt.1740082500.pcap.gz
```

so the flow-aggregation step lives with us. This notebook does it:

1. lists the pcaps under one S3 prefix — by default **a whole month** — and checks
   per-day coverage (§3),
2. **streams** each one (S3 body → gzip → pcap records; nothing is downloaded to disk or
   held whole in memory) and aggregates packets into **FlowTuple records**, in parallel
   across cores and resumable across jobs (§4–5),
3. writes one Parquet file per pcap (§5), and
4. queries the Parquet files with PySpark and reports on them: daily trends, unique source / destination addresses,
   protocol mix, TCP destination and source ports, TCP flags, UDP ports, ICMP types,
   top sources, destination coverage, TTLs, packet sizes and **source countries** (§6–13).

### Which "FlowTuple"

A FlowTuple record (corsaro) is one row per distinct key per time interval, with a packet
count:

| field | meaning |
|---|---|
| `time` | start of the interval (unix seconds; 60 s bins by default, as corsaro uses) |
| `src_ip`, `dst_ip` | IPv4 addresses as unsigned 32-bit integers (as in the Avro FlowTuple) |
| `src_port`, `dst_port` | TCP/UDP ports; for ICMP, **type** and **code** |
| `protocol` | IP protocol number (6 TCP, 17 UDP, 1 ICMP, …) |
| `ttl`, `tcp_flags`, `ip_len` | IP TTL, TCP flag byte, IP total length (wire length, so unaffected by snaplen) |
| `tcp_synlen`, `tcp_synwinlen` | TCP header length and window of the flow's first SYN (0 otherwise) |
| `packet_cnt` | packets that matched this key in this interval |
| `is_masscan` | SYN carries masscan's signature `ip_id == (dst_ip ^ dst_port ^ seq) & 0xffff` |

This is the **per-destination (v3-style) layout**, chosen on purpose: the FlowTuple *v4*
records the darknet module reads have already rolled destinations and source ports up into
`dst_net` + `uniq_*` counters, which would make the destination-address and source-port
questions below unanswerable. v4 is a `groupby` away from this table; the reverse is not
possible.

**Source country** (FlowTuple's `maxmind_country`) is not a Parquet column: it is joined
on at analysis time from the MaxMind GeoLite2 database (§2b, §6), the same database
`nids-telescope-traffic/flow_analysis.ipynb` uses. Keeping it out of the files means a
different database edition never forces a re-conversion.

**Not produced here** (they need other external datasets): `prefix2asn`,
`netacq_country`, and `is_spoofed`. The darknet module's `build_prefix_to_asn` in
`nids-telescope-traffic` is the pattern to add the first.

Runs on **Expanse**; read-only against the object store; prints no credential values.

## 0 · Dependencies

Same import-then-`pip --user` pattern as `greynet_data_access.ipynb`. `dpkt` is used only
for its streaming pcap record reader; headers are unpacked by hand (§4). `maxminddb`
reads the GeoLite2 geolocation database (§2b). The analysis
half uses PySpark from the container (via `findspark`), not a pip install — it needs a JVM.

In [ ]:
import importlib
import site
import subprocess
import sys


def ensure(pkg, module=None):
    "Import `module` (default: pkg), pip-installing --user first if missing."
    module = module or pkg
    try:
        importlib.import_module(module)
        print(f"{pkg:<10} OK")
        return
    except ImportError:
        pass
    print(f"{pkg:<10} missing — installing with pip --user ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--quiet", pkg])
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())
    importlib.invalidate_caches()
    importlib.import_module(module)
    print(f"{pkg:<10} installed")


ensure("boto3")
ensure("dpkt")
ensure("maxminddb")
ensure("pandas")
ensure("pyarrow")
ensure("matplotlib")

## 1 · Configuration

`PREFIX` selects which pcaps to convert. The bucket is partitioned
`datasource=/year=/month=/day=/hour=/` with one pcap per 5-minute window, so the default
**month prefix is up to 12 × 24 × 28 = 8,064 pcaps**. That changes how this has to run:

- **Conversion is parallel and resumable.** `N_WORKERS` processes each stream one pcap at
  a time; a pcap whose Parquet already exists is skipped, so a job that hits its time
  limit is simply resubmitted and carries on where it stopped.
- **Run it as a batch job**, not an interactive session, with as many cores as you can
  get — conversion is CPU-bound, roughly 150k packets/s per core. To size the job, first
  run with `MAX_PCAPS = 12` (one hour): §5 prints the measured throughput and an
  estimate for the whole selection.
- **Analysis is PySpark in local mode**, the same single-node pattern as the darknet module
  (`nids-ucsdnt-expanse`): give the job enough memory, because one JVM holds everything.
- **Output goes to Lustre scratch**, not the repo directory — a month of FlowTuples will
  not fit in a home-directory quota.

`GEOIP_DB` is the MaxMind GeoLite2 database used to geolocate source addresses — the
same object `nids-telescope-traffic/flow_analysis.ipynb` reads. It may be a local path,
an http(s) URL, or a list of those tried in order; `None` turns geolocation off.

`LOCAL_PCAPS` bypasses S3 entirely: put paths to `.pcap` / `.pcap.gz` files there to run
the same pipeline on local captures.

In [ ]:
import getpass
import os

ENV_FILE = "/expanse/lustre/projects/sdp178/kmok/.ucsdnts3greynet.env"

BUCKET = "telescope-sdscnt-pcap-live"
PREFIX = "datasource=sdsc-nt/year=2025/month=02/"   # one month; append "day=20/" or "day=20/hour=20/" to narrow
MAX_PCAPS = None         # None = everything under PREFIX; an integer = only the first N (in time order)

LOCAL_PCAPS = []         # e.g. ["/scratch/.../sdscnt.1740082500.pcap.gz"]; non-empty => S3 is skipped

# Parallelism: the cores this job was actually given (Slurm cgroup), not the node's total.
N_WORKERS = len(os.sched_getaffinity(0))

INTERVAL = 60            # FlowTuple time bin, seconds
OVERWRITE = False        # False: a pcap whose Parquet already exists is not re-parsed

# Where the FlowTuple Parquet files go (mirrors the bucket's key layout underneath).
_scratch = f"/expanse/lustre/scratch/{getpass.getuser()}/temp_project"
OUT_DIR = os.environ.get("FLOWTUPLE_DIR") or (
    os.path.join(_scratch, "greynet_flowtuples") if os.path.isdir(_scratch) else "flowtuples"
)

TOP_N = 20               # rows in the "top ..." tables

# MaxMind GeoLite2 City database for source geolocation (§2b): a path, an http(s) URL, or a
# list tried in order. Same object as nids-telescope-traffic/flow_analysis.ipynb: first by
# its in-cluster name (resolves only inside NRP), then by the public NRP gateway, which is
# what works from Expanse. None = no geolocation.
_GEOIP_KEY = "caida/geolocation/maxmind/2026-06-24.GeoLite2-City.mmdb.gz"
GEOIP_DB = os.environ.get("GEOIP_DB") or [
    f"http://rook-ceph-rgw-nautiluss3.rook/{_GEOIP_KEY}",
    f"https://s3-west.nrp-nautilus.io/{_GEOIP_KEY}",
]

# Spark (analysis half). local[N] runs driver and executors in ONE JVM, so driver memory is the
# real ceiling; None = 70% of what this job was allocated. Must be set before Spark starts.
SPARK_DRIVER_MEMORY = None   # e.g. "48g"

# Same gateway and override rules as greynet_data_access.ipynb.
DEFAULT_ENDPOINT = "https://hermes.caida.org"
ENDPOINT_OVERRIDE = None

print(f"workers : {N_WORKERS}")
print(f"output  : {os.path.abspath(OUT_DIR)}")

## 2 · Credentials and S3 client

Identical to `greynet_data_access.ipynb` §2–3: parse the shell-style env file,
auto-detect the access / secret / endpoint variables, print only their **names**.
Skipped when `LOCAL_PCAPS` is set.

In [ ]:
import pathlib
import re


def parse_env_file(path):
    "Parse a shell-style env file: optional `export`, KEY=value, quotes, # comments."
    env = {}
    for line in pathlib.Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        line = re.sub(r"^export\s+", "", line)
        m = re.match(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$", line)
        if not m:
            continue
        key, val = m.group(1), m.group(2).strip()
        if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
            val = val[1:-1]
        env[key] = val
    return env


def pick(env, *patterns):
    "First env key matching any regex, tried in order (case-insensitive)."
    for pat in patterns:
        hits = [k for k in env if re.search(pat, k, re.I)]
        if hits:
            return hits[0]
    return None


s3 = None
if LOCAL_PCAPS:
    print("LOCAL_PCAPS set — skipping S3.")
else:
    import boto3
    from botocore.config import Config

    try:
        env = parse_env_file(ENV_FILE)
        print(f"parsed {ENV_FILE}")
    except OSError as exc:
        print(f"could not read {ENV_FILE} ({exc}); falling back to kernel environment")
        env = {k: v for k, v in os.environ.items() if re.search(r"s3|greynet", k, re.I)}

    access_var = pick(env, r"access.?key.?id", r"access.?key", r"access")
    secret_var = pick(env, r"secret.?access.?key", r"secret.?key", r"secret")
    endpoint_var = pick(env, r"endpoint", r"\burl\b", r"host")
    assert access_var and secret_var, (
        f"Could not identify access/secret variables among: {sorted(env)} — "
        "set access_var / secret_var manually in this cell."
    )
    S3_ENDPOINT_URL = (
        ENDPOINT_OVERRIDE or (env.get(endpoint_var) if endpoint_var else None) or DEFAULT_ENDPOINT
    )
    print(f"access key variable : {access_var}")
    print(f"secret key variable : {secret_var}  (value not shown)")
    print(f"endpoint            : {S3_ENDPOINT_URL}")

    def make_s3():
        "A fresh client. Each conversion worker process builds its own (§5)."
        return boto3.client(
            "s3",
            endpoint_url=S3_ENDPOINT_URL,
            aws_access_key_id=env[access_var],
            aws_secret_access_key=env[secret_var],
            config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
        )

    s3 = make_s3()

## 2b · GeoLite2 country database (MaxMind)

Fetched here, before the long conversion, so an unreachable database stops the job in its
first minute rather than after hours of parsing.

**Pulling it** is the telescope notebook's `build_geoip_reader`, unchanged in substance:
open the path or URL, sniff the gzip magic instead of trusting `.gz`, and read the whole
`.mmdb` into memory (`MODE_FD`) — it is only tens of MB.

**Using it** differs, because of scale. The telescope notebook looks each unique source
IP up in Python; a month of greynet traffic has far too many for that. Instead the
database is flattened **once** into a table of IPv4 ranges → country code
(`country.iso_code`, the same field `geolocate_ip` returns), which Spark joins against in
§6. The table is cached as Parquet under `OUT_DIR/geoip/`, so resubmitted jobs skip both
the download and the flattening.

Flattening walks the database's search tree network by network
(`get_with_prefix_len`), one /8 per task across the worker processes; neighbouring
networks in the same country are merged. Each range is then listed once per /16 it
touches (`bucket`), which gives Spark an equality key to join on.

The database is a snapshot (see the build date printed below) and address assignments
move over time, so countries for captures from a different period are approximate.


In [ ]:
import concurrent.futures
import contextlib
import datetime
import gzip
import io
import ipaddress
import multiprocessing
import urllib.parse
import urllib.request

import maxminddb
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


def _open_binary_stream(source):
    "Local path or http(s) URL -> raw byte stream (not decompressed yet)."
    if urllib.parse.urlparse(source).scheme in ("http", "https"):
        return urllib.request.urlopen(source, timeout=120)
    return open(source, "rb")


def open_geoip_db(source):
    "Open a MaxMind .mmdb (optionally gzipped), fully in memory. Close it when done."
    with contextlib.closing(_open_binary_stream(source)) as raw:
        buffered = raw if hasattr(raw, "peek") else io.BufferedReader(raw)
        is_gzip = buffered.peek(2)[:2] == b"\x1f\x8b"
        stream = gzip.GzipFile(fileobj=buffered) if is_gzip else buffered
        return maxminddb.open_database(stream, mode=maxminddb.MODE_FD)


_geo_reader = None  # set in the parent; the fork()ed workers below inherit it


def _country_ranges_in_slash8(first_octet):
    "Walk one /8 of the open database: [[first ip, last ip, country code], ...]."
    lo = first_octet << 24
    hi = lo | 0xFFFFFF
    out = []
    ip = lo
    while ip <= hi:
        rec, prefix_len = _geo_reader.get_with_prefix_len(str(ipaddress.IPv4Address(ip)))
        end = min(ip | ((1 << (32 - prefix_len)) - 1), hi)  # last address of the network holding ip
        cc = ((rec or {}).get("country") or {}).get("iso_code")
        if cc:
            if out and out[-1][2] == cc and out[-1][1] + 1 == ip:
                out[-1][1] = end  # extends the previous range
            else:
                out.append([ip, end, cc])
        ip = end + 1
    return out


def build_country_ranges(source, out_path):
    "Flatten the database at `source` into the range table at `out_path`."
    global _geo_reader
    _geo_reader = open_geoip_db(source)
    try:
        meta = _geo_reader.metadata()
        if N_WORKERS > 1:
            ctx = multiprocessing.get_context("fork")
            with concurrent.futures.ProcessPoolExecutor(N_WORKERS, mp_context=ctx) as pool:
                chunks = list(pool.map(_country_ranges_in_slash8, range(256), chunksize=2))
        else:
            chunks = [_country_ranges_in_slash8(o) for o in range(256)]
    finally:
        _geo_reader.close()
        _geo_reader = None
    rows = [r for chunk in chunks for r in chunk]
    assert rows, f"no IPv4 country records in {source}"
    start = np.array([r[0] for r in rows], dtype=np.int64)
    end = np.array([r[1] for r in rows], dtype=np.int64)
    # One row per (range, /16 it touches).
    per_range = (end >> 16) - (start >> 16) + 1
    idx = np.repeat(np.arange(len(rows)), per_range)
    nth = np.arange(len(idx)) - np.repeat(np.cumsum(per_range) - per_range, per_range)
    table = pa.table(
        {
            "bucket": (start >> 16)[idx] + nth,
            "ip_start": start[idx],
            "ip_end": end[idx],
            "country": pa.array([r[2] for r in rows], type=pa.string()).take(pa.array(idx)),
        }
    ).replace_schema_metadata(
        {"geoip_source": source, "database_type": meta.database_type, "build_epoch": str(meta.build_epoch)}
    )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    pq.write_table(table, out_path + ".partial", compression="zstd")
    os.replace(out_path + ".partial", out_path)


def _geo_cache_path(source):
    name = re.sub(r"\.mmdb(\.gz)?$", "", os.path.basename(urllib.parse.urlparse(source).path))
    return os.path.join(OUT_DIR, "geoip", f"{name}.country_ranges.parquet")


GEO_RANGES = None  # Parquet path of the range table; None = geolocation off
if not GEOIP_DB:
    print("GEOIP_DB is None — geolocation is off; country columns below will be empty.")
else:
    _candidates = [GEOIP_DB] if isinstance(GEOIP_DB, str) else list(GEOIP_DB)
    GEO_RANGES = next((p for p in map(_geo_cache_path, _candidates) if os.path.exists(p)), None)
    if GEO_RANGES:
        print(f"using cached range table {GEO_RANGES}")
    else:
        _errors = []
        for _source in _candidates:
            try:
                build_country_ranges(_source, _geo_cache_path(_source))
            except OSError as exc:  # unreachable URL / missing file: try the next candidate
                _errors.append(f"  {_source}: {type(exc).__name__}: {exc}")
                continue
            GEO_RANGES = _geo_cache_path(_source)
            print(f"read {_source}")
            break
        if not GEO_RANGES:
            raise RuntimeError(
                "Could not read the GeoLite2 database from any GEOIP_DB location:\n" + "\n".join(_errors)
                + "\nPoint GEOIP_DB (or the GEOIP_DB environment variable) at a reachable copy, "
                "or set it to None to run without geolocation."
            )

    _g = pq.read_table(GEO_RANGES)
    _meta = {k.decode(): v.decode() for k, v in (_g.schema.metadata or {}).items()}
    _r = _g.select(["ip_start", "ip_end", "country"]).to_pandas().drop_duplicates("ip_start")
    _built = datetime.datetime.fromtimestamp(int(_meta["build_epoch"]), datetime.timezone.utc)
    print(f"{_meta['database_type']} built {_built:%Y-%m-%d}: {len(_r):,} IPv4 ranges "
          f"({_g.num_rows:,} rows by /16), {_r['country'].nunique()} countries, "
          f"covering {100 * (_r['ip_end'] - _r['ip_start'] + 1).sum() / 2**32:.1f}% of IPv4 space")


## 3 · Select the pcaps and check coverage

Lists every pcap under `PREFIX` and tabulates them **per day**. A full day is 288
five-minute files; days with fewer are flagged here, before any parsing, because every
monthly total below is only as complete as this listing. (The window's start time is
taken from the file name, `sdscnt.<unix time>.pcap.gz`.)

In [ ]:
import datetime

import pandas as pd


def out_path_for(rel):
    "Parquet path for a pcap, mirroring its key / relative path under OUT_DIR."
    return os.path.join(OUT_DIR, re.sub(r"\.pcap(\.gz)?$", "", rel) + ".flowtuple.parquet")


def _file_ts(name):
    m = re.search(r"\.(\d{9,10})\.", name)
    return int(m.group(1)) if m else None


sources = []
if LOCAL_PCAPS:
    for path in LOCAL_PCAPS:
        name = os.path.basename(path)
        sources.append({"name": name, "size": os.path.getsize(path), "path": path, "key": None,
                        "out": out_path_for(name)})
else:
    objs = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
        objs.extend(o for o in page.get("Contents", []) if ".pcap" in o["Key"])
    objs.sort(key=lambda o: o["Key"])
    print(f"{len(objs):,} pcap objects under s3://{BUCKET}/{PREFIX}")
    for o in objs[:MAX_PCAPS]:
        sources.append({"name": o["Key"].rsplit("/", 1)[-1], "size": o["Size"], "path": None, "key": o["Key"],
                        "out": out_path_for(o["Key"])})

assert sources, "No pcaps selected — check PREFIX / LOCAL_PCAPS."
total_gib = sum(s["size"] for s in sources) / 2**30
print(f"selected {len(sources):,} pcaps, {total_gib:,.2f} GiB compressed")

cov = pd.DataFrame(
    {
        "day": [
            datetime.datetime.fromtimestamp(t, datetime.timezone.utc).strftime("%Y-%m-%d") if t else "unknown"
            for t in (_file_ts(s["name"]) for s in sources)
        ],
        "GiB": [s["size"] / 2**30 for s in sources],
    }
)
coverage = cov.groupby("day").agg(pcaps=("GiB", "size"), GiB=("GiB", "sum")).round({"GiB": 2})
coverage["pct_of_288"] = (100 * coverage["pcaps"] / 288).round(1)
short = coverage[coverage["pcaps"] < 288]
print(f"{len(coverage)} day(s); {len(short)} with fewer than 288 pcaps"
      + (f": {', '.join(short.index[:10])}{' ...' if len(short) > 10 else ''}" if len(short) else ""))
display(coverage)

## 4 · Packets → FlowTuple records

**Reading.** The byte stream is sniffed for the gzip magic (never trusting `.gz`),
decompressed incrementally, and handed to `dpkt.pcap.Reader`, which yields one
`(timestamp, frame bytes)` pair at a time.

**Parsing.** Only fixed-offset header fields are needed, so they are unpacked directly
with `struct` instead of building dpkt's per-packet object graph — several times faster,
which matters at tens of millions of packets. The link layer is taken from the pcap
header rather than assumed: Ethernet (with any 802.1Q / QinQ tags skipped), raw IP, and
Linux cooked capture are handled; anything else stops with a clear error.

**What is counted, what is not** — every packet lands in exactly one bucket, printed per
file in §5 so nothing disappears silently:

- `ipv4` — aggregated into FlowTuples.
- `non_ipv4` — ARP, IPv6, etc. (tallied by ethertype). IPv6 is out of scope, as in
  FlowTuple.
- `truncated` — capture cut off before the fields we need (snaplen or a malformed header).

Non-first IP fragments carry no transport header; they are kept with ports and flags 0
and counted under `fragments`.

**Memory.** The key includes the time bin, and each time the capture advances into a new
bin the finished bins are flushed to the Parquet file as a row group. Peak memory is one
interval's worth of distinct flows, independent of how long the capture is.

In [ ]:
import collections
import gzip
import io
import struct
import time as _time

import dpkt
import pyarrow as pa
import pyarrow.parquet as pq

FLOWTUPLE_SCHEMA = pa.schema(
    [
        ("time", pa.int64()),
        ("src_ip", pa.uint32()),
        ("dst_ip", pa.uint32()),
        ("src_port", pa.uint16()),
        ("dst_port", pa.uint16()),
        ("protocol", pa.uint8()),
        ("ttl", pa.uint8()),
        ("tcp_flags", pa.uint8()),
        ("ip_len", pa.uint16()),
        ("tcp_synlen", pa.uint16()),
        ("tcp_synwinlen", pa.uint16()),
        ("packet_cnt", pa.int64()),
        ("is_masscan", pa.bool_()),
    ]
)
_KEY_COLS = FLOWTUPLE_SCHEMA.names[:9]
_VAL_COLS = FLOWTUPLE_SCHEMA.names[9:]

DLT_EN10MB, DLT_RAW, DLT_RAW_ALT, DLT_LINUX_SLL = 1, 12, 101, 113
_VLAN_ETHERTYPES = (0x8100, 0x88A8, 0x9100)
_IP_HDR = struct.Struct("!HHHBBHII")  # total len, id, flags+frag off, ttl, proto, cksum, src, dst
_TCP_HDR = struct.Struct("!HHIIBBH")  # sport, dport, seq, ack, data offset, flags, window
_UDP_PORTS = struct.Struct("!HH")


def open_pcap_stream(raw):
    "Raw byte stream -> buffered, transparently gunzipped stream."
    buffered = io.BufferedReader(raw, 1 << 20) if not hasattr(raw, "peek") else raw
    if buffered.peek(2)[:2] == b"\x1f\x8b":
        return io.BufferedReader(gzip.GzipFile(fileobj=buffered), 1 << 20)
    return buffered


def _flush(writer, flows):
    "Write a {key: [synlen, synwin, count, masscan]} dict as one Parquet row group."
    if not flows:
        return 0
    keys = list(zip(*flows.keys()))
    vals = list(zip(*flows.values()))
    arrays = [pa.array(col, type=FLOWTUPLE_SCHEMA.field(name).type) for name, col in zip(_KEY_COLS, keys)]
    arrays += [pa.array(col, type=FLOWTUPLE_SCHEMA.field(name).type) for name, col in zip(_VAL_COLS, vals)]
    writer.write_table(pa.Table.from_arrays(arrays, schema=FLOWTUPLE_SCHEMA))
    return len(flows)


def pcap_to_flowtuple(raw, out_path, interval=60):
    "Stream one pcap into a FlowTuple Parquet file. Returns a dict of run statistics."
    stream = open_pcap_stream(raw)
    reader = dpkt.pcap.Reader(stream)
    dlt = reader.datalink()
    if dlt not in (DLT_EN10MB, DLT_RAW, DLT_RAW_ALT, DLT_LINUX_SLL):
        raise ValueError(f"unsupported pcap link type {dlt} — add its header length to pcap_to_flowtuple")

    n_packets = n_ipv4 = n_truncated = n_fragments = n_records = wire_bytes = 0
    non_ipv4 = collections.Counter()
    first_ts = last_ts = None
    flows = {}
    cur_bin = None
    unpack_ip, unpack_tcp, unpack_udp = _IP_HDR.unpack_from, _TCP_HDR.unpack_from, _UDP_PORTS.unpack_from

    tmp_path = f"{out_path}.partial"
    t0 = _time.time()
    with pq.ParquetWriter(tmp_path, FLOWTUPLE_SCHEMA, compression="zstd") as writer:
        for ts, buf in reader:
            n_packets += 1
            try:
                # --- link layer -> offset of the IP header ---
                if dlt == DLT_EN10MB:
                    etype = (buf[12] << 8) | buf[13]
                    off = 14
                    while etype in _VLAN_ETHERTYPES:
                        etype = (buf[off + 2] << 8) | buf[off + 3]
                        off += 4
                elif dlt == DLT_LINUX_SLL:
                    etype = (buf[14] << 8) | buf[15]
                    off = 16
                else:
                    etype = 0x0800 if buf[0] >> 4 == 4 else 0x86DD if buf[0] >> 4 == 6 else 0
                    off = 0
                if etype != 0x0800:
                    non_ipv4[etype] += 1
                    continue

                # --- IPv4 ---
                vihl = buf[off]
                if vihl >> 4 != 4:
                    n_truncated += 1
                    continue
                ip_len, ip_id, frag, ttl, proto, _, src, dst = unpack_ip(buf, off + 2)
                l4 = off + (vihl & 0x0F) * 4

                # --- transport ---
                sport = dport = flags = synlen = synwin = 0
                masscan = False
                if frag & 0x1FFF:  # non-first fragment: no transport header
                    n_fragments += 1
                elif proto == 6:
                    sport, dport, seq, _, doff, flags, win = unpack_tcp(buf, l4)
                    if flags & 0x02:  # SYN
                        synlen = (doff >> 4) * 4
                        synwin = win
                        masscan = ip_id == (dst ^ dport ^ seq) & 0xFFFF
                elif proto == 17:
                    sport, dport = unpack_udp(buf, l4)
                elif proto == 1:
                    sport, dport = buf[l4], buf[l4 + 1]  # ICMP type, code
            except (IndexError, struct.error):
                n_truncated += 1
                continue

            n_ipv4 += 1
            wire_bytes += ip_len
            ts = float(ts)
            if first_ts is None:
                first_ts = ts
            last_ts = ts

            tbin = int(ts) // interval * interval
            if cur_bin is None:
                cur_bin = tbin
            elif tbin > cur_bin:  # capture moved into a new interval: flush finished ones
                n_records += _flush(writer, flows)
                flows = {}
                cur_bin = tbin

            key = (tbin, src, dst, sport, dport, proto, ttl, flags, ip_len)
            rec = flows.get(key)
            if rec is None:
                flows[key] = [synlen, synwin, 1, masscan]
            else:
                rec[2] += 1
        n_records += _flush(writer, flows)
    os.replace(tmp_path, out_path)  # a killed run never leaves a half-written file under the final name

    elapsed = _time.time() - t0
    return {
        "datalink": dlt,
        "snaplen": reader.snaplen,
        "packets": n_packets,
        "ipv4": n_ipv4,
        "non_ipv4": sum(non_ipv4.values()),
        "non_ipv4_ethertypes": {f"0x{k:04x}": v for k, v in non_ipv4.most_common(5)},
        "truncated": n_truncated,
        "fragments": n_fragments,
        "flowtuple_records": n_records,
        "ipv4_wire_bytes": wire_bytes,
        "first_ts": first_ts,
        "last_ts": last_ts,
        "parse_seconds": round(elapsed, 1),
        "packets_per_sec": round(n_packets / elapsed) if elapsed else None,
    }

## 5 · Run the conversion

Each worker process converts one pcap at a time and writes, next to the Parquet file, a
small `.stats.json` with that file's packet accounting — so the totals below cover
skipped (already converted) files too, and survive across resubmitted jobs.

A pcap that fails (unreadable object, corrupt gzip, …) is recorded and the run
continues; failures are listed at the end and are **not** silently dropped from the
coverage figures. Re-running the cell retries only what is missing.

In [ ]:
import concurrent.futures
import json
import multiprocessing

_worker_s3 = {}  # one client per worker process; a boto3 client must not be shared across fork()


def _open_source(src):
    if src["path"]:
        return open(src["path"], "rb")
    pid = os.getpid()
    if pid not in _worker_s3:
        _worker_s3[pid] = make_s3()
    return _worker_s3[pid].get_object(Bucket=BUCKET, Key=src["key"])["Body"]


def convert_one(src):
    "Convert one pcap (or load its stats if already done). Never raises."
    out = src["out"]
    stats_path = out + ".stats.json"
    if not OVERWRITE and os.path.exists(out) and os.path.exists(stats_path):
        with open(stats_path) as fh:
            return {"pcap": src["name"], "status": "skipped", **json.load(fh)}
    raw = None
    try:
        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
        raw = _open_source(src)
        st = pcap_to_flowtuple(raw, out, INTERVAL)
    except Exception as exc:  # keep the batch alive; report below
        for leftover in (out + ".partial",):
            if os.path.exists(leftover):
                os.remove(leftover)
        return {"pcap": src["name"], "status": "FAILED", "error": f"{type(exc).__name__}: {exc}"}
    finally:
        if raw is not None:
            raw.close()
    with open(stats_path, "w") as fh:
        json.dump(st, fh)
    return {"pcap": src["name"], "status": "converted", **st}


t_start = _time.time()
results = []
report_every = max(1, len(sources) // 40)


def _progress(res):
    results.append(res)
    if res["status"] == "FAILED":
        print(f"  FAILED {res['pcap']}: {res['error']}", flush=True)
    if len(results) % report_every == 0 or len(results) == len(sources):
        el = _time.time() - t_start
        eta = el / len(results) * (len(sources) - len(results))
        print(f"  {len(results):>6,} / {len(sources):,}   elapsed {el / 60:6.1f} min   eta {eta / 60:6.1f} min", flush=True)


if N_WORKERS > 1 and len(sources) > 1:
    ctx = multiprocessing.get_context("fork")  # workers inherit the functions defined in this notebook
    with concurrent.futures.ProcessPoolExecutor(min(N_WORKERS, len(sources)), mp_context=ctx) as pool:
        for fut in concurrent.futures.as_completed([pool.submit(convert_one, s) for s in sources]):
            _progress(fut.result())
else:
    for s in sources:
        _progress(convert_one(s))

res = pd.DataFrame(results)
done = res[res["status"] != "FAILED"]
failed = res[res["status"] == "FAILED"]
converted = res[res["status"] == "converted"]
assert len(done), "Every pcap failed — see the errors above."


def _utc(ts):
    return datetime.datetime.fromtimestamp(ts, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if ts else "-"


flowtuple_files = sorted(s["out"] for s in sources if os.path.exists(s["out"]))
parquet_gib = sum(os.path.getsize(p) for p in flowtuple_files) / 2**30

conv = pd.Series(
    {
        "pcaps selected": f"{len(sources):,}",
        "  converted this run": f"{len(converted):,}",
        "  already converted (skipped)": f"{int((res['status'] == 'skipped').sum()):,}",
        "  FAILED": f"{len(failed):,}",
        "packets read": f"{int(done['packets'].sum()):,}",
        "  ipv4 aggregated": f"{int(done['ipv4'].sum()):,}",
        "    of which non-first fragments": f"{int(done['fragments'].sum()):,}",
        "  non-ipv4 skipped": f"{int(done['non_ipv4'].sum()):,}",
        "  truncated": f"{int(done['truncated'].sum()):,}",
        "flowtuple records": f"{int(done['flowtuple_records'].sum()):,}",
        "packets per record": f"{done['ipv4'].sum() / max(done['flowtuple_records'].sum(), 1):.2f}",
        "capture span (UTC)": f"{_utc(done['first_ts'].min())} -> {_utc(done['last_ts'].max())}",
        "link types / snaplens seen": f"{sorted(set(done['datalink']))} / {sorted(set(done['snaplen']))}",
        "parquet size (GiB)": f"{parquet_gib:,.2f}  (pcaps: {total_gib:,.2f} compressed)",
    },
    name="value",
)
display(conv.to_frame())

if len(converted):
    cpu_s = converted["parse_seconds"].sum()
    rate = converted["packets"].sum() / max(cpu_s, 1e-9)
    conv_bytes = sum(s["size"] for s in sources if s["name"] in set(converted["pcap"]))
    print(f"throughput: {rate:,.0f} packets/s per core; {cpu_s / 3600:.2f} core-hours for "
          f"{conv_bytes / 2**30:,.2f} GiB of pcap this run")
    if conv_bytes:
        print(f"=> a selection of {total_gib:,.1f} GiB needs about {cpu_s / conv_bytes * total_gib * 2**30 / 3600:,.1f} "
              f"core-hours ({cpu_s / conv_bytes * total_gib * 2**30 / 3600 / N_WORKERS:,.1f} h wall on {N_WORKERS} workers)")
if len(failed):
    print(f"\n{len(failed)} pcap(s) FAILED and are missing from everything below:")
    display(failed[["pcap", "error"]].set_index("pcap"))

---
# Empirical analysis

Everything below reads **only the Parquet files**, through **PySpark** — a month of
per-destination FlowTuples is far more rows than fit in a pandas DataFrame, so each table
is one Spark SQL aggregation that returns a small result. The session follows the darknet
module (`nids-ucsdnt-expanse`): `local[N]` on one node, driver memory set explicitly,
Arrow off, and small results brought back with `.collect()`.

Reading the numbers:

- "packets" figures are `packet_cnt`-weighted; "records" are FlowTuple rows.
- Every unique-address count is **exact** (no sketches). For "top N" tables the distinct
  counts are computed only for the N rows shown, which is what keeps that affordable.
- Totals cover the pcaps that converted. Check §3 (missing files) and §5 (failures)
  before quoting a monthly figure.

Destination addresses here are the real greynet addresses, so tables below report
destinations as **counts and /24 aggregates**, not as individual addresses — keep it that
way if this notebook's output is going to be committed or shared.

## 6 · Load and overview

In [ ]:
import ipaddress

import decimal

import matplotlib.pyplot as plt
import numpy as np

try:
    import findspark  # locates the container's Spark install, as nids-ucsdnt-expanse does

    findspark.init()
except Exception:
    pass  # no findspark / SPARK_HOME: fall back to a pip-installed pyspark
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def _job_memory_mb():
    "Memory this job was allocated (Slurm), else the machine's physical RAM."
    if os.environ.get("SLURM_MEM_PER_NODE"):
        return int(os.environ["SLURM_MEM_PER_NODE"])
    if os.environ.get("SLURM_MEM_PER_CPU"):
        return int(os.environ["SLURM_MEM_PER_CPU"]) * N_WORKERS
    return os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") // 2**20


driver_memory = SPARK_DRIVER_MEMORY or f"{int(_job_memory_mb() * 0.7)}m"

# Shuffle spill: the job's node-local SSD if there is one, otherwise next to the output.
_node_scratch = f"/scratch/{getpass.getuser()}/job_{os.environ.get('SLURM_JOB_ID', '')}"
spark_local_dir = _node_scratch if os.path.isdir(_node_scratch) else os.path.join(OUT_DIR, "spark_tmp")
os.makedirs(spark_local_dir, exist_ok=True)

conf = pyspark.SparkConf().setAll(
    [
        ("spark.master", f"local[{N_WORKERS}]"),
        ("spark.app.name", "greynet flowtuple analysis"),
        # Left unset this is ~1g and the JVM is OOM-killed ("SparkContext was shut down").
        ("spark.driver.memory", driver_memory),
        ("spark.driver.maxResultSize", "4g"),
        ("spark.local.dir", spark_local_dir),
        ("spark.sql.shuffle.partitions", str(max(64, 8 * N_WORKERS))),
        ("spark.sql.session.timeZone", "UTC"),
        ("spark.sql.execution.arrow.pyspark.enabled", "false"),
        ("spark.ui.showConsoleProgress", "false"),
    ]
)
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version}  master={spark.sparkContext.master}  driver memory={driver_memory}  spill dir={spark_local_dir}")

spark.read.parquet(*flowtuple_files).createOrReplaceTempView("ft")

# Source geolocation: the §2b range table (empty when geolocation is off), and `ft_geo` = ft
# plus a src_country column. The table is small, so it is broadcast and the join shuffles
# nothing; ranges never overlap, so it cannot duplicate a FlowTuple record (§12 checks that).
geo_df = (
    spark.read.parquet(GEO_RANGES) if GEO_RANGES
    else spark.createDataFrame([], "bucket long, ip_start long, ip_end long, country string")
)
geo_df.createOrReplaceTempView("geo")
GEO_ON = "shiftright(src_ip, 16) = bucket AND src_ip BETWEEN ip_start AND ip_end"
(
    spark.table("ft")
    .join(F.broadcast(geo_df), F.expr(GEO_ON), "left")
    .drop("bucket", "ip_start", "ip_end")
    .withColumnRenamed("country", "src_country")
    .createOrReplaceTempView("ft_geo")
)


def q(sql):
    '''Run Spark SQL and return a (small) pandas DataFrame.

    Uses .collect(), not .toPandas(): the Arrow conversion crashes in this runtime
    (see nids-ucsdnt-expanse's PySpark-Parquet guide). Only aggregated results come back.
    '''
    sdf = spark.sql(sql)
    df = pd.DataFrame(sdf.collect(), columns=sdf.columns)
    for c in df.columns:  # Spark DECIMAL -> float
        if df[c].dtype == object and df[c].map(lambda v: isinstance(v, decimal.Decimal)).any():
            df[c] = df[c].astype(float)
    return df


def n_distinct(expr, where="TRUE"):
    "Exact number of distinct values of a SQL expression (de-duplicate, then count)."
    return int(q(f"SELECT count(*) AS n FROM (SELECT DISTINCT {expr} AS v FROM ft WHERE {where})").iloc[0]["n"])


def materialize(name, sql):
    "Run a large aggregation once, park the result as Parquet, and expose it as a view."
    path = os.path.join(OUT_DIR, "analysis_tmp", name)
    spark.sql(sql).write.mode("overwrite").parquet(path)
    spark.read.parquet(path).createOrReplaceTempView(name)


PROTO_NAMES = {1: "ICMP", 6: "TCP", 17: "UDP", 47: "GRE", 50: "ESP", 41: "IPv6-in-IPv4", 132: "SCTP"}
PROTO_SQL = ("CASE protocol " + " ".join(f"WHEN {k} THEN '{v}'" for k, v in PROTO_NAMES.items())
             + " ELSE CAST(protocol AS STRING) END")
BYTES_SQL = "CAST(ip_len AS BIGINT) * packet_cnt"


def breakdown(groups, where="TRUE", uniq=("src_ip", "dst_ip"), n=None, extra=""):
    '''Packets (and share) per group, plus exact distinct counts of each column in `uniq`.

    groups: {output name: SQL expression}. With n, only the top-n groups by packets are
    returned, and the distinct counts are computed for those groups only. Distinct counts
    are a de-duplicating GROUP BY followed by a count, which Spark runs as an ordinary shuffle that can spill to disk.
    '''
    names = list(groups)
    by = ", ".join(str(i + 1) for i in range(len(names)))
    sel = ", ".join(f"{expr} AS {name}" for name, expr in groups.items())
    base = q(f'''
        SELECT {sel}, sum(packet_cnt) AS packets,
               round(100 * sum(packet_cnt) / sum(sum(packet_cnt)) OVER (), 2) AS pct_packets{extra}
        FROM ft WHERE {where} GROUP BY {by} ORDER BY packets DESC {f"LIMIT {n}" if n else ""}
    ''')
    keep = ""
    if n and len(base):
        rows = ", ".join("(" + ", ".join(str(int(v)) for v in row) + ")" for row in base[names].itertuples(index=False))
        keep = f" AND ({', '.join(groups.values())}) IN ({rows})"
    for col in uniq if len(base) else ():
        u = q(f'''
            SELECT {', '.join(names)}, count(*) AS uniq_{col}s
            FROM (SELECT DISTINCT {sel}, {col} FROM ft WHERE {where}{keep}) GROUP BY {by}
        ''')
        base = base.merge(u, on=names, how="left")
    for col in uniq if not len(base) else ():
        base[f"uniq_{col}s"] = []
    return base.set_index(names)


def ip_str(v):
    return str(ipaddress.ip_address(int(v)))


def net_str(v, bits):
    return f"{ip_str(int(v) << (32 - bits))}/{bits}"


o = q(f'''
    SELECT count(*) AS records, sum(packet_cnt) AS packets, sum({BYTES_SQL}) AS bytes,
           min(time) AS t_first, max(time) AS t_last,
           coalesce(sum(packet_cnt) FILTER (WHERE is_masscan), 0) AS masscan
    FROM ft
''').iloc[0].to_dict()
# Exact distinct counts, one de-duplicating pass each.
o.update(
    bins=n_distinct("time"),
    src_ips=n_distinct("src_ip"), src_24s=n_distinct("shiftright(src_ip, 8)"),
    dst_ips=n_distinct("dst_ip"), dst_24s=n_distinct("shiftright(dst_ip, 8)"),
    dst_16s=n_distinct("shiftright(dst_ip, 16)"),
)

total_packets = int(o["packets"])
t_first, t_last = int(o["t_first"]), int(o["t_last"]) + INTERVAL
covered_s = int(o["bins"]) * INTERVAL  # seconds that actually have data (gaps excluded)

overview = pd.Series(
    {
        "flowtuple files": f"{len(flowtuple_files):,}",
        "time range (UTC)": f"{_utc(t_first)} -> {_utc(t_last)}",
        "time covered by data": f"{covered_s / 86400:,.2f} days ({100 * covered_s / (t_last - t_first):.1f}% of the range)",
        "flowtuple records": f"{int(o['records']):,}",
        "packets": f"{total_packets:,}",
        "bytes (IP total length)": f"{int(o['bytes']):,}",
        "mean packets / s (over covered time)": f"{total_packets / covered_s:,.1f}",
        "mean packet size (bytes)": f"{o['bytes'] / total_packets:,.1f}",
        "unique source IPs": f"{int(o['src_ips']):,}",
        "unique source /24s": f"{int(o['src_24s']):,}",
        "unique destination IPs": f"{int(o['dst_ips']):,}",
        "unique destination /24s": f"{int(o['dst_24s']):,}",
        "unique destination /16s": f"{int(o['dst_16s']):,}",
        "packets flagged masscan": f"{int(o['masscan']):,}",
    },
    name="value",
)
display(overview.to_frame())

## 7 · Protocol mix and traffic over time

Per protocol: packets, bytes, records, and how many distinct sources and destinations
it involves. Packets and distinct sources often rank protocols differently — a handful
of heavy scanners can dominate packets while most *sources* arrive over another protocol.

Over a month the time axis matters as much as the totals: the rate plot shows whether
the mix is steady or driven by a few events, and the per-day table shows how many
sources appear each day (the month's unique-source total in §6 is much larger than any
one day's, because most sources are short-lived).

In [ ]:
proto_tbl = breakdown(
    {"proto": PROTO_SQL}, extra=f", sum({BYTES_SQL}) AS bytes, count(*) AS records"
)
display(proto_tbl)

In [ ]:
# Chart style: one restrained look for every figure below.
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
PROTO_COLORS = {"TCP": BLUE, "UDP": ORANGE, "ICMP": AQUA}  # colour follows the protocol, everywhere
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": "#c9c8c2", "axes.labelcolor": "#52514e",
        "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
        "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "font.size": 9, "figure.dpi": 110,
    }
)

# Plot bin: the finest of 1 min / 5 min / 1 h / 1 day that keeps the series under ~800 points.
TS_BIN = next(b for b in (INTERVAL, 300, 3600, 86400) if b >= INTERVAL and (t_last - t_first) / b <= 800 or b == 86400)
TS_LABEL = {60: "1 min", 300: "5 min", 3600: "1 h", 86400: "1 day"}.get(TS_BIN, f"{TS_BIN} s")

ts = q(f'''
    SELECT time - time % {TS_BIN} AS t,
           CASE WHEN protocol IN (6, 17, 1) THEN {PROTO_SQL} ELSE 'other' END AS proto,
           sum(packet_cnt) AS packets
    FROM ft GROUP BY 1, 2
''')
# Rate over the seconds that have data in each plot bin, so a partly-missing hour is not drawn as a dip.
bins_per_t = q(f"SELECT t, count(*) AS bins FROM (SELECT DISTINCT time - time % {TS_BIN} AS t, time FROM ft) GROUP BY 1").set_index("t")["bins"]
per_bin = ts.pivot_table(index="t", columns="proto", values="packets", aggfunc="sum")
per_bin = per_bin.reindex(range(int(per_bin.index.min()), int(per_bin.index.max()) + 1, TS_BIN))  # gaps -> NaN -> line breaks
rate = per_bin.div(bins_per_t.reindex(per_bin.index) * INTERVAL, axis=0)
x = pd.to_datetime(rate.index, unit="s", utc=True)
marker = dict(marker="o", ms=4) if len(rate) <= 40 else {}

fig, ax = plt.subplots(figsize=(10, 3.4))
for p, color in (("TCP", BLUE), ("UDP", ORANGE), ("ICMP", AQUA), ("other", GRAY)):
    if p in rate:
        ax.plot(x, rate[p], color=color, lw=1.6 if len(rate) > 40 else 2, label=p, **marker)
ax.set_yscale("log")
ax.set_ylabel("packets / s (log)")
ax.set_xlabel(f"time (UTC), {TS_LABEL} bins — gaps are missing data")
ax.set_title("Packet rate by protocol")
ax.legend(frameon=False, ncol=4, loc="upper left", bbox_to_anchor=(0, -0.22))
fig.autofmt_xdate(rotation=0, ha="center")
plt.show()

In [ ]:
daily = breakdown(
    {"day": "from_unixtime(time - time % 86400, 'yyyy-MM-dd')"},
    uniq=("time", "src_ip", "dst_ip"),
    extra=f''', sum({BYTES_SQL}) AS bytes,
        round(100 * coalesce(sum(packet_cnt) FILTER (WHERE protocol = 6), 0) / sum(packet_cnt), 1) AS pct_tcp,
        round(100 * coalesce(sum(packet_cnt) FILTER (WHERE protocol = 17), 0) / sum(packet_cnt), 1) AS pct_udp,
        round(100 * coalesce(sum(packet_cnt) FILTER (WHERE protocol = 1), 0) / sum(packet_cnt), 1) AS pct_icmp''',
).sort_index()
daily.insert(0, "hours_of_data", (daily.pop("uniq_times") * INTERVAL / 3600).round(1))
daily.insert(2, "packets_per_s", (daily["packets"] / (daily["hours_of_data"] * 3600).clip(lower=INTERVAL)).round(1))
daily = daily.drop(columns="pct_packets")
daily = daily[["hours_of_data", "packets", "packets_per_s", "bytes", "uniq_src_ips", "uniq_dst_ips", "pct_tcp", "pct_udp", "pct_icmp"]]
display(daily)

if len(daily) > 1:
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
    xd = pd.to_datetime(daily.index)
    axes[0].bar(xd, daily["packets_per_s"], width=0.8, color=BLUE)
    axes[0].set_title("Mean packet rate per day (packets / s, over the hours that have data)")
    axes[1].bar(xd, daily["uniq_src_ips"], width=0.8, color=BLUE)
    axes[1].set_title("Unique source IPs per day")
    for ax in axes:
        ax.grid(axis="x", visible=False)
        ax.ticklabel_format(axis="y", style="plain")
    axes[1].xaxis.set_major_locator(plt.matplotlib.dates.AutoDateLocator(minticks=3, maxticks=16))
    axes[1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%m-%d"))
    fig.tight_layout()
    plt.show()

## 8 · Source and destination addresses

**Sources** — who is sending. The top-sources table shows, per source, how many
distinct greynet addresses and TCP/UDP ports it touched (many destinations × one port is a
horizontal scan; few destinations × many ports is a vertical scan) and on how many
separate days it was seen — persistent scanners versus one-off visitors — plus its
GeoLite2 country (§12 has the per-country view).

**Destinations** — how evenly the monitored addresses are hit. In a darknet, coverage is
close to uniform; in a greynet, unevenness across addresses is one of the things worth
comparing.

In [ ]:
# One row per source IP, built once and parked as Parquet; everything in this section reads
# that small table, not the FlowTuples. Distinct counts are de-duplicating GROUP BYs joined back on.
materialize("src_agg", '''
    WITH p AS (SELECT src_ip, sum(packet_cnt) AS packets FROM ft GROUP BY 1),
         d AS (SELECT src_ip, count(*) AS uniq_dst_ips FROM (SELECT DISTINCT src_ip, dst_ip FROM ft) GROUP BY 1),
         r AS (SELECT src_ip, count(*) AS uniq_dst_ports FROM (SELECT DISTINCT src_ip, dst_port FROM ft WHERE protocol IN (6, 17)) GROUP BY 1),
         t AS (SELECT src_ip, count(*) AS days_seen FROM (SELECT DISTINCT src_ip, time DIV 86400 AS day FROM ft) GROUP BY 1)
    SELECT p.src_ip, p.packets, d.uniq_dst_ips, coalesce(r.uniq_dst_ports, 0) AS uniq_dst_ports, t.days_seen
    FROM p JOIN d ON p.src_ip = d.src_ip JOIN t ON p.src_ip = t.src_ip LEFT JOIN r ON p.src_ip = r.src_ip
''')
# The same table with each source's GeoLite2 country (NULL = not in the database).
materialize("src_geo", f'''
    SELECT /*+ BROADCAST(geo) */ s.*, geo.country
    FROM src_agg s LEFT JOIN geo ON {GEO_ON}
''')
materialize("dst_agg", '''
    WITH p AS (SELECT dst_ip, sum(packet_cnt) AS packets FROM ft GROUP BY 1),
         u AS (SELECT dst_ip, count(*) AS uniq_src_ips FROM (SELECT DISTINCT dst_ip, src_ip FROM ft) GROUP BY 1)
    SELECT p.dst_ip, p.packets, u.uniq_src_ips FROM p JOIN u ON p.dst_ip = u.dst_ip
''')

QS = "array(0.5, 0.9, 0.99, 1.0)"
sq = q(f"SELECT percentile(packets, {QS}) AS a, percentile(uniq_dst_ips, {QS}) AS b, "
       f"percentile(days_seen, {QS}) AS c FROM src_agg").iloc[0]
dq = q(f"SELECT percentile(packets, {QS}) AS a, percentile(uniq_src_ips, {QS}) AS b FROM dst_agg").iloc[0]
dist = pd.DataFrame(
    {
        "packets per source IP": sq["a"],
        "destination IPs per source IP": sq["b"],
        "days a source IP was seen": sq["c"],
        "packets per destination IP": dq["a"],
        "source IPs per destination IP": dq["b"],
    },
    index=["median", "p90", "p99", "max"],
)
display(dist.T)

s = q('''
    SELECT count(*) AS n, count(*) FILTER (WHERE packets = 1) AS one_packet,
           count(*) FILTER (WHERE days_seen = 1) AS one_day
    FROM src_agg
''').iloc[0]
top_src = q(f"SELECT src_ip, coalesce(country, '-') AS country, packets, uniq_dst_ips, uniq_dst_ports, days_seen "
            f"FROM src_geo ORDER BY packets DESC LIMIT {TOP_N}")
top10 = top_src["packets"].nlargest(10).sum()
protos = q(f'''
    SELECT src_ip, concat_ws(',', sort_array(collect_set({PROTO_SQL}))) AS protocols
    FROM ft WHERE src_ip IN ({", ".join(str(int(v)) for v in top_src["src_ip"])}) GROUP BY 1
''')
print(f"sources that sent exactly one packet    : {int(s['one_packet']):,} of {int(s['n']):,} ({100 * s['one_packet'] / s['n']:.1f}%)")
print(f"sources seen on only one day            : {int(s['one_day']):,} ({100 * s['one_day'] / s['n']:.1f}%)")
print(f"share of packets from the top 10 sources: {100 * top10 / total_packets:.1f}%")

top_src = top_src.merge(protos, on="src_ip", how="left")
top_src.insert(3, "pct_packets", (100 * top_src["packets"] / total_packets).round(2))
top_src["src_ip"] = top_src["src_ip"].map(ip_str)
display(top_src.set_index("src_ip"))

In [ ]:
# Destinations: /24 aggregates only (see the note at the top of the analysis).
dst24 = breakdown({"dst_net24": "shiftright(dst_ip, 8)"}, uniq=("dst_ip", "src_ip")).reset_index()
dst24 = dst24.rename(columns={"uniq_dst_ips": "dst_ips_hit"})
n_dst24 = len(dst24)
top_dst24 = dst24.head(TOP_N).copy()
top_dst24["dst_net24"] = top_dst24["dst_net24"].map(lambda v: net_str(v, 24))
print(f"{n_dst24:,} destination /24s; top {len(top_dst24)} by packets:")
display(top_dst24.set_index("dst_net24"))

# Fan-out: how many greynet addresses does each source touch?
fan = q("SELECT uniq_dst_ips AS x, count(*) AS n FROM src_agg GROUP BY 1 ORDER BY 1")
ccdf = fan["n"][::-1].cumsum()[::-1] / fan["n"].sum()  # fraction of sources with fan-out >= x
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.step(fan["x"], ccdf, where="mid", color=BLUE, lw=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("distinct destination IPs contacted by a source (log)")
ax.set_ylabel("fraction of sources ≥ x (log)")
ax.set_title("Source fan-out (CCDF)")
plt.show()

## 9 · TCP — flags, destination ports, source ports

**Flags first**, because they decide how to read the ports:

- **SYN only** — connection attempts: scanning / probing. The **destination** port is the
  service being looked for; the source port is ephemeral and mostly noise.
- **SYN+ACK, RST, RST+ACK** — *backscatter*: replies from hosts that were sent packets
  spoofed with a greynet source address (DoS victims, typically). Here the **source**
  port is the attacked service, and the destination port is the noise.

So destination ports are tabulated over SYN-only traffic and all TCP, and source ports
are shown split by scan vs. backscatter.

In [ ]:
TCP_FLAG_BITS = ((0x02, "S"), (0x10, "A"), (0x01, "F"), (0x04, "R"), (0x08, "P"), (0x20, "U"), (0x40, "E"), (0x80, "C"))


def tcp_flags_str(flags):
    return "".join(letter for bit, letter in TCP_FLAG_BITS if flags & bit) or "."


SYN_ONLY = "protocol = 6 AND tcp_flags = 2"                      # S
BACKSCATTER = "protocol = 6 AND tcp_flags IN (18, 4, 20)"        # SA, R, RA

flag_tbl = breakdown({"tcp_flags": "tcp_flags"}, where="protocol = 6", uniq=("src_ip",)).reset_index()
flag_tbl = flag_tbl.rename(columns={"pct_packets": "pct_tcp_packets"})
flag_tbl.insert(0, "flags", flag_tbl["tcp_flags"].map(tcp_flags_str))
display(flag_tbl.drop(columns="tcp_flags").set_index("flags").head(12))

t = q(f'''
    SELECT coalesce(sum(packet_cnt), 0) AS tcp,
           coalesce(sum(packet_cnt) FILTER (WHERE tcp_flags = 2), 0) AS syn,
           coalesce(sum(packet_cnt) FILTER (WHERE tcp_flags = 2 AND is_masscan), 0) AS syn_masscan,
           coalesce(sum(packet_cnt) FILTER (WHERE tcp_flags IN (18, 4, 20)), 0) AS backscatter
    FROM ft WHERE protocol = 6
''').iloc[0].to_dict()
t.update(dports=n_distinct("dst_port", "protocol = 6"), sports=n_distinct("src_port", "protocol = 6"))
print(f"TCP packets            : {int(t['tcp']):,}")
print(f"  SYN-only (scanning)  : {int(t['syn']):,}   of which masscan-signature: {int(t['syn_masscan']):,}")
print(f"  SA / R / RA (backscatter-like): {int(t['backscatter']):,}")
print(f"distinct TCP destination ports: {int(t['dports']):,}   distinct TCP source ports: {int(t['sports']):,}")

In [ ]:
SERVICES = {
    21: "ftp", 22: "ssh", 23: "telnet", 25: "smtp", 53: "dns", 80: "http", 81: "http-alt", 110: "pop3",
    123: "ntp", 135: "msrpc", 137: "netbios-ns", 139: "netbios", 143: "imap", 161: "snmp", 389: "ldap",
    443: "https", 445: "smb", 465: "smtps", 500: "ike", 587: "submission", 993: "imaps", 995: "pop3s",
    1433: "mssql", 1521: "oracle", 1900: "ssdp", 2222: "ssh-alt", 2323: "telnet-alt", 3306: "mysql",
    3389: "rdp", 3478: "stun", 5060: "sip", 5353: "mdns", 5432: "postgres", 5555: "adb", 5900: "vnc",
    6379: "redis", 7547: "tr-069", 8080: "http-alt", 8443: "https-alt", 8728: "mikrotik", 9200: "elasticsearch",
    11211: "memcached", 27017: "mongodb", 37215: "huawei-hg532", 52869: "realtek-upnp",
}


def port_table(where, col, n=TOP_N):
    "Top-n values of a port column by packets (share is of all packets matching `where`)."
    g = breakdown({col: col}, where=where, n=n)
    g.insert(0, "service", [SERVICES.get(int(p), "") for p in g.index])
    return g


print("TCP destination ports — SYN-only (what scanners are looking for):")
tcp_dport_syn = port_table(SYN_ONLY, "dst_port")
display(tcp_dport_syn)
print("TCP destination ports — all TCP:")
display(port_table("protocol = 6", "dst_port"))

In [ ]:
if len(tcp_dport_syn):
    d = tcp_dport_syn.iloc[::-1]
    labels = [f"{p}  {s}".rstrip() for p, s in zip(d.index, d["service"])]
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.28 * len(d) + 1.2), sharey=True)
    for ax, col, title in (
        (axes[0], "packets", "SYN packets"),
        (axes[1], "uniq_src_ips", "distinct source IPs"),
    ):
        ax.barh(labels, d[col], color=BLUE, height=0.7)
        ax.set_title(title)
        ax.grid(axis="y", visible=False)
        ax.ticklabel_format(axis="x", style="plain")
    axes[0].set_ylabel("TCP destination port")
    fig.suptitle(f"Top {len(d)} scanned TCP ports — by volume, and by how many sources ask", x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("no SYN-only TCP traffic")

In [ ]:
print("TCP source ports — SYN-only (scanner side; fixed source ports are a tool fingerprint):")
display(port_table(SYN_ONLY, "src_port"))
print("TCP source ports — backscatter-like SA / R / RA (the service that was attacked):")
tcp_sport_bs = port_table(BACKSCATTER, "src_port")
display(tcp_sport_bs)

## 10 · UDP ports and ICMP types

For ICMP the FlowTuple `src_port` / `dst_port` fields carry **type** / **code**
(8/0 echo request = ping scanning; 0/0 echo reply, 3/x unreachable, 11/x time exceeded
= backscatter).

In [ ]:
u = q("SELECT coalesce(sum(packet_cnt), 0) AS packets FROM ft WHERE protocol = 17").iloc[0].to_dict()
u.update(dports=n_distinct("dst_port", "protocol = 17"), sports=n_distinct("src_port", "protocol = 17"))
print(f"UDP packets: {int(u['packets']):,}   distinct destination ports: {int(u['dports']):,}"
      f"   distinct source ports: {int(u['sports']):,}")
print("UDP destination ports:")
udp_dport = port_table("protocol = 17", "dst_port")
display(udp_dport)
print("UDP source ports:")
display(port_table("protocol = 17", "src_port"))

ICMP_TYPES = {0: "echo reply", 3: "dest unreachable", 5: "redirect", 8: "echo request", 11: "time exceeded",
              13: "timestamp request", 14: "timestamp reply", 17: "address mask request"}
icmp_tbl = breakdown({"icmp_type": "src_port", "icmp_code": "dst_port"}, where="protocol = 1", n=TOP_N).reset_index()
icmp_tbl.insert(2, "meaning", icmp_tbl["icmp_type"].map(ICMP_TYPES).fillna(""))
print("ICMP type / code:")
display(icmp_tbl.set_index(["icmp_type", "icmp_code"]))

## 11 · TTL and packet sizes

Observed TTL = sender's initial TTL (almost always 64, 128, or 255) minus hop count, so
rounding up to the next of those gives a rough OS-family / tool split (64 Linux & most
IoT, 128 Windows, 255 network gear and some raw-socket scanners). Packet sizes are
dominated by a few values: 40 = bare TCP SYN with no options (stateless scanners such as
masscan / ZMap), 44–60 = SYN with options (real OS stacks).

In [ ]:
ttl = q("SELECT ttl, sum(packet_cnt) AS packets FROM ft GROUP BY 1").set_index("ttl")["packets"]
ttl = ttl.reindex(range(256), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(ttl.index, ttl.values, width=1.0, color=BLUE)
for mark in (64, 128, 255):
    ax.axvline(mark + 0.5, color=GRAY, lw=0.8, ls=":")
    ax.text(mark, ax.get_ylim()[1], f" {mark}", va="top", ha="right", color="#52514e")
ax.set_xlim(0, 256)
ax.set_xlabel("IP TTL as received")
ax.set_ylabel("packets")
ax.ticklabel_format(axis="y", style="plain")
ax.set_title("TTL distribution")
plt.show()

ttl_tbl = breakdown(
    {"ttl_range": "CASE WHEN ttl <= 32 THEN '1: ≤32' WHEN ttl <= 64 THEN '2: 33–64 (init 64)' "
                  "WHEN ttl <= 128 THEN '3: 65–128 (init 128)' ELSE '4: 129–255 (init 255)' END"},
    uniq=("src_ip",),
).sort_index().reset_index()
ttl_tbl["ttl_range"] = ttl_tbl["ttl_range"].str[3:]
display(ttl_tbl.set_index("ttl_range"))

size_tbl = q(f'''
    SELECT {PROTO_SQL} AS proto, ip_len, sum(packet_cnt) AS packets,
           round(100 * sum(packet_cnt) / sum(sum(packet_cnt)) OVER (), 2) AS pct_packets
    FROM ft GROUP BY 1, 2 ORDER BY packets DESC LIMIT {TOP_N}
''')
print("most common IP packet lengths:")
display(size_tbl.set_index(["proto", "ip_len"]))

syn_fp = breakdown({"tcp_synlen": "tcp_synlen", "tcp_synwinlen": "tcp_synwinlen"}, where=SYN_ONLY, uniq=("src_ip",), n=10)
syn_fp = syn_fp.rename(columns={"pct_packets": "pct_syn_packets"})
print("most common SYN (TCP header length, window) fingerprints:")
display(syn_fp)

## 12 · Source countries

Each source address is mapped to a country with the GeoLite2 table from §2b. Read it with
three cautions:

- **It locates the address, not the actor.** For scanning (SYN-only) the source is the
  scanner — or the proxy / cloud host it rents. For backscatter the source is the
  *victim* of a spoofed-source attack. The traffic-mix columns below are there so the two
  are not confused.
- **`unknown`** means GeoLite2 has no country for the address (unallocated, reserved or
  private space — often spoofed), not that the lookup failed.
- **Packets and sources rank countries differently.** `pct_from_top_src` shows how much of
  a country's packets came from its single busiest address; a high value means the
  country's rank is really one host's.

The first table is built from the per-source table of §8, so it costs nothing; the
traffic mix and the daily series take one more pass over the FlowTuples.


In [ ]:
if not GEO_RANGES:
    print("geolocation is off (GEOIP_DB is None) — nothing to report")
else:
    country_tbl = q('''
        SELECT coalesce(country, 'unknown') AS country,
               sum(packets) AS packets,
               round(100 * sum(packets) / sum(sum(packets)) OVER (), 2) AS pct_packets,
               count(*) AS uniq_src_ips,
               round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_src_ips,
               round(100 * max(packets) / sum(packets), 1) AS pct_from_top_src
        FROM src_geo GROUP BY 1 ORDER BY packets DESC
    ''').set_index("country")
    assert int(country_tbl["packets"].sum()) == total_packets, "geo join changed the packet total"
    unk = country_tbl.reindex(["unknown"]).fillna(0).iloc[0]
    print(f"{len(country_tbl.drop(index='unknown', errors='ignore'))} countries; no GeoLite2 country for "
          f"{int(unk['uniq_src_ips']):,} source IPs ({unk['pct_src_ips']:.2f}%) sending {unk['pct_packets']:.2f}% of packets")
    display(country_tbl.head(TOP_N))

    d = country_tbl.head(TOP_N).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.28 * len(d) + 1.2), sharey=True)
    for ax, col, title in (
        (axes[0], "packets", "packets"),
        (axes[1], "uniq_src_ips", "distinct source IPs"),
    ):
        ax.barh(d.index, d[col], color=BLUE, height=0.7)
        ax.set_title(title)
        ax.grid(axis="y", visible=False)
        ax.ticklabel_format(axis="x", style="plain")
    axes[0].set_ylabel("source country")
    fig.suptitle(f"Top {len(d)} source countries — by volume, and by number of sources", x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    plt.show()


In [ ]:
if GEO_RANGES:
    # One pass over the FlowTuples: packets per (country, day), split by what kind of traffic it is.
    cd = q(f'''
        SELECT coalesce(src_country, 'unknown') AS country,
               from_unixtime(time - time % 86400, 'yyyy-MM-dd') AS day,
               sum(packet_cnt) AS packets,
               coalesce(sum(packet_cnt) FILTER (WHERE {SYN_ONLY}), 0) AS tcp_syn,
               coalesce(sum(packet_cnt) FILTER (WHERE {BACKSCATTER}), 0) AS tcp_backscatter,
               coalesce(sum(packet_cnt) FILTER (WHERE protocol = 6), 0) AS tcp,
               coalesce(sum(packet_cnt) FILTER (WHERE protocol = 17), 0) AS udp,
               coalesce(sum(packet_cnt) FILTER (WHERE protocol = 1), 0) AS icmp
        FROM ft_geo GROUP BY 1, 2
    ''')
    assert int(cd["packets"].sum()) == total_packets, "geo join changed the packet total"

    mix = cd.drop(columns="day").groupby("country").sum().sort_values("packets", ascending=False).head(TOP_N)
    mix_pct = pd.DataFrame(
        {
            "packets": mix["packets"],
            "pct_tcp_syn": 100 * mix["tcp_syn"] / mix["packets"],
            "pct_tcp_backscatter": 100 * mix["tcp_backscatter"] / mix["packets"],
            "pct_tcp_other": 100 * (mix["tcp"] - mix["tcp_syn"] - mix["tcp_backscatter"]) / mix["packets"],
            "pct_udp": 100 * mix["udp"] / mix["packets"],
            "pct_icmp": 100 * mix["icmp"] / mix["packets"],
        }
    ).round(1)
    print("traffic mix per source country (% of that country's packets):")
    display(mix_pct)

    by_day = cd.pivot_table(index="day", columns="country", values="packets", aggfunc="sum").sort_index()
    if len(by_day) > 1:
        # Same normalisation as §7: mean rate over the hours of the day that have data.
        rate = by_day.div(daily["hours_of_data"].reindex(by_day.index) * 3600, axis=0)
        top5 = [c for c in mix.index if c != "unknown"][:5]
        fig, ax = plt.subplots(figsize=(10, 3.4))
        xd = pd.to_datetime(rate.index)
        for c, color in zip(top5, (BLUE, ORANGE, AQUA, "#8a5cc7", "#c79a1e")):
            ax.plot(xd, rate[c], color=color, lw=1.6, label=c, **(dict(marker="o", ms=4) if len(rate) <= 40 else {}))
        ax.set_yscale("log")
        ax.set_ylabel("packets / s (log)")
        ax.set_xlabel("day (UTC)")
        ax.set_title(f"Daily mean packet rate, top {len(top5)} source countries")
        ax.legend(frameon=False, ncol=5, loc="upper left", bbox_to_anchor=(0, -0.22))
        ax.xaxis.set_major_locator(plt.matplotlib.dates.AutoDateLocator(minticks=3, maxticks=16))
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%m-%d"))
        plt.show()


## 13 · Summary

In [ ]:
def _top(tbl, label):
    if not len(tbl):
        return f"{label}: none"
    items = [f"{idx}{'/' + r['service'] if r['service'] else ''} ({r['pct_packets']:.1f}%)" for idx, r in tbl.head(5).iterrows()]
    return f"{label}: " + ", ".join(items)


print(f"{len(flowtuple_files):,} pcaps, {_utc(t_first)} -> {_utc(t_last)} UTC "
      f"({covered_s / 86400:.2f} days of data; {len(failed)} pcap(s) failed to convert)")
print(f"{total_packets:,} IPv4 packets in {int(o['records']):,} FlowTuple records "
      f"({total_packets / covered_s:,.0f} pkt/s)")
print(f"{int(o['src_ips']):,} source IPs -> {int(o['dst_ips']):,} destination IPs in {n_dst24:,} /24s")
print("protocols: " + ", ".join(f"{p} {r['pct_packets']:.1f}%" for p, r in proto_tbl.head(4).iterrows()))
print(_top(tcp_dport_syn, "top scanned TCP ports (share of SYNs)"))
print(_top(tcp_sport_bs, "top TCP backscatter source ports"))
print(_top(udp_dport, "top UDP destination ports (share of UDP)"))
if GEO_RANGES:
    print("top source countries: " + ", ".join(f"{c} ({r['pct_packets']:.1f}%)" for c, r in country_tbl.head(5).iterrows()))


## Next steps

- **Normalize by monitored addresses.** `sdscgreynetlist` gives the greynet address set;
  join it against `dst_ip` to get packets per monitored address (the number comparable to
  the darknet) and to see whether any destinations fall *outside* the list.
- **Enrich sources with `prefix2asn`** (RouteViews RIB, as in
  `nids-telescope-traffic/flow_analysis.ipynb`): the same range-table join used for
  countries here works for prefixes, and splits each country's traffic by network.
- **More months.** Change `PREFIX` and resubmit; converted files are kept under `OUT_DIR`
  with the bucket's `year=/month=/day=/hour=` layout, so several months can be analysed
  together by pointing Spark at the directory. If one node is too slow, split the days
  across a Slurm array — every pcap is independent.
- **Roll up to FlowTuple v4** if the darknet comparison needs identical records:
  `GROUP BY time, src_ip, shiftright(dst_ip, 8), dst_port, protocol` with distinct counters.